# DiaRisk — Step 1: Basic Data Analysis

Exploratory analysis of the **Pima Indians Diabetes** dataset (UCI).

**Goal:** understand feature distributions, class balance, and data quality before model training.

## 1. Imports and load data

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw" / "pima-indians-diabetes.csv"

COLUMNS = [
    "pregnancies",
    "glucose",
    "blood_pressure",
    "skin_thickness",
    "insulin",
    "bmi",
    "diabetes_pedigree",
    "age",
    "outcome",
]

df = pd.read_csv(RAW, header=None, names=COLUMNS)
df.head()

,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,age,outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 2. Shape and dtypes

How many rows/columns? Are types numeric?

In [2]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print()
df.info()

Rows: 768, Columns: 9

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   pregnancies        768 non-null    int64  
 1   glucose            768 non-null    int64  
 2   blood_pressure     768 non-null    int64  
 3   skin_thickness     768 non-null    int64  
 4   insulin            768 non-null    int64  
 5   bmi                768 non-null    float64
 6   diabetes_pedigree  768 non-null    float64
 7   age                768 non-null    int64  
 8   outcome            768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


## 3. Target balance (`outcome`)

- `0` = no diabetes
- `1` = diabetes

Imbalance matters later when we choose metrics (not only accuracy).

In [3]:
counts = df["outcome"].value_counts().sort_index()
percents = df["outcome"].value_counts(normalize=True).sort_index() * 100

summary = pd.DataFrame({"count": counts, "percent": percents.round(1)})
summary.index = summary.index.map({0: "no diabetes (0)", 1: "diabetes (1)"})
display(summary)

ax = counts.plot(kind="bar", color=["#4C78A8", "#F58518"], rot=0)
ax.set_title("Target balance")
ax.set_xlabel("outcome")
ax.set_ylabel("count")
ax.set_xticklabels(["no diabetes", "diabetes"])
plt.tight_layout()
plt.show()

,count,percent
outcome,,
no diabetes (0),500,65.1
diabetes (1),268,34.9


/var/folders/99/mmmwv9m51yv94lvxpsw9k69c0000gn/T/ipykernel_6302/2935544238.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Descriptive statistics

Mean, std, min/max — first feel for the feature ranges.

In [4]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
pregnancies,768.0,3.85,3.37,0.00,1.00,3.00,6.00,17.00
glucose,768.0,120.89,31.97,0.00,99.00,117.00,140.25,199.00
blood_pressure,768.0,69.11,19.36,0.00,62.00,72.00,80.00,122.00
skin_thickness,768.0,20.54,15.95,0.00,0.00,23.00,32.00,99.00
insulin,768.0,79.80,115.24,0.00,0.00,30.50,127.25,846.00
bmi,768.0,31.99,7.88,0.00,27.30,32.00,36.60,67.10
diabetes_pedigree,768.0,0.47,0.33,0.08,0.24,0.37,0.63,2.42
age,768.0,33.24,11.76,21.00,24.00,29.00,41.00,81.00
outcome,768.0,0.35,0.48,0.00,0.00,0.00,1.00,1.00


## 5. Zeros that may mean missing

In this dataset, `0` is often **not physiologically realistic** for some fields  
(e.g. glucose, BMI, blood pressure) and usually means **missing**.

We only **observe** this here. Cleaning comes in a later step.

In [5]:
zero_cols = ["glucose", "blood_pressure", "skin_thickness", "insulin", "bmi"]
zero_counts = pd.Series({col: int((df[col] == 0).sum()) for col in zero_cols})
zero_counts = zero_counts.to_frame("n_zeros")
zero_counts["percent"] = (100 * zero_counts["n_zeros"] / len(df)).round(1)
zero_counts

,n_zeros,percent
glucose,5,0.7
blood_pressure,35,4.6
skin_thickness,227,29.6
insulin,374,48.7
bmi,11,1.4


## 6. Feature distributions

Histograms for each feature, colored by `outcome`.

In [6]:
feature_cols = [c for c in df.columns if c != "outcome"]

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.ravel()

for ax, col in zip(axes, feature_cols):
    sns.histplot(
        data=df,
        x=col,
        hue="outcome",
        bins=20,
        element="step",
        stat="density",
        common_norm=False,
        ax=ax,
        palette={0: "#4C78A8", 1: "#F58518"},
    )
    ax.set_title(col)
    ax.set_xlabel("")

fig.suptitle("Feature distributions by outcome", y=1.02)
plt.tight_layout()
plt.show()

/var/folders/99/mmmwv9m51yv94lvxpsw9k69c0000gn/T/ipykernel_6302/3516620592.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Correlation heatmap

Which features move together? Which relate most to `outcome`?

In [7]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True)
plt.title("Correlation heatmap")
plt.tight_layout()
plt.show()

print("Correlation with outcome (sorted):")
print(corr["outcome"].drop("outcome").sort_values(ascending=False).round(3))

Correlation with outcome (sorted):
glucose              0.467
bmi                  0.293
age                  0.238
pregnancies          0.222
diabetes_pedigree    0.174
insulin              0.131
skin_thickness       0.075
blood_pressure       0.065
Name: outcome, dtype: float64


/var/folders/99/mmmwv9m51yv94lvxpsw9k69c0000gn/T/ipykernel_6302/234079747.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Takeaways for the next step

1. **768 rows**, 8 clinical features + binary diabetes outcome.
2. Classes are **imbalanced** (~65% / 35%) → use precision/recall/ROC, not only accuracy.
3. Some zeros are likely **missing values** → handle in the training pipeline.
4. **Glucose** and **BMI** typically correlate most with the outcome.

**Next (Step 2):** train a Logistic Regression baseline, then compare with stronger models.